# 4 · Pitch grids and export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/EN/jupyter/markov-book-en/04_pitch_and_export.ipynb)

So far the chain could only produce notes it had seen. Now we shape the **pitch material** with
GreasyPidgin's `PitchGrid`, and export the result as MIDI, sheet music (MusicXML), CSV and JSON.

In [ ]:
# Setup -- runs by itself (about 20 seconds the first time)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # which version of GreasyPidgin to use (a git branch or tag)
TUTORIAL_REF  = "main"   # which version of this tutorial's helper file to use
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/EN/jupyter/markov-book-en/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

print("Ready!")

In [ ]:
from GreasyPidgin import PitchGrid

midi_path = wh.choose_midi()
melody = wh.load_melody(midi_path)

chain = MarkovChain()
chain.generateRulesFromList(melody.pitches, order=2)
new_pitches = chain.getNElements(seed=tuple(melody.pitches[:2]), n=32)

wh.render_midi(new_pitches, melody.deltas, melody.bpm, "original_scale.mid")
print("A melody from your chain, as in chapter 2:")
wh.listen("original_scale.mid")

## Which scale is it in?

A `PitchGrid` is the set of allowed pitches. `fromCorrelation` compares the notes of your melody
with a list of scales and returns the best fit as a grid. We let it choose among a few common scales
and all twelve possible roots.

In [ ]:
scale = PitchGrid.fromCorrelation(
    melody.pitches, 36, 96,
    possibleSystems    = ["edo12"],
    possibleMasks      = ["ionian", "aeolian", "dorian", "mixolydian", "majorPentatonic", "minorPentatonic"],
    possibleScaleRoots = list(range(60, 72)),
)
print("best fit:", wh.pitch_class_name(scale.scaleRoot), scale.maskName)
print("pitches :", [wh.pitch_name(p) for p in sorted(scale)[:14]], "...")

## Move the melody into another scale

`quantise` moves every note to the nearest pitch of the grid. Here: C minor (aeolian) —
the same melody with its third, sixth and seventh pulled down.

In [ ]:
other = PitchGrid.fromMask("edo12", "aeolian", 36, 96, "c4", "c4")

in_minor = [other.quantise(float(p)) for p in new_pitches]
print(" ".join(wh.pitch_name(p) for p in in_minor))

wh.render_midi(in_minor, melody.deltas, melody.bpm, "in_minor.mid")
wh.listen("in_minor.mid")

**Try it:** replace `"aeolian"` by `"dorian"`, `"phrygian"`, `"lydian"`, `"wholeTone"`,
`"majorPentatonic"` or `"harmonicMinor"`.

## Beyond the piano: other tunings

GreasyPidgin knows many tuning systems. In a historical tuning such as Werckmeister I, the scale degrees
are not exactly on the piano keys, so the pitches are **fractional MIDI numbers**.
When exported, they become MIDI *pitch-wheel* messages (and the exporter writes the deviation in cents
as a text label).

In [ ]:
werck = PitchGrid.fromMask("werckmeister1", "ionian", 36, 96, "c4", "c4")
detuned = [werck.quantise(float(p)) for p in new_pitches]

print([round(p, 2) for p in detuned[:10]], "...")

## Export

Everything above was built as a **Gesture**. A Gesture can write itself in several formats.
We use the minor-scale version:

In [ ]:
gesture = wh.render_midi(in_minor, melody.deltas, melody.bpm, "melody.mid")

gesture.toCsv("melody.csv")      # a table: one row per note
gesture.toJson("melody.json")    # all details as structured data

In [ ]:
# MusicXML needs the library music21 (installed here only if it is missing)
import importlib.util, subprocess, sys
if importlib.util.find_spec("music21") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "music21"], check=True)

gesture.toXml("melody.xml", title="A Markov melody")

| file | for |
|---|---|
| `melody.mid` | any DAW, notation program or synthesizer |
| `melody.xml` | MuseScore, Dorico, Finale, Sibelius: opens as sheet music |
| `melody.csv` | Excel / pandas: look at (or edit) the numbers |
| `melody.json` | other programs and scripts |

All four files in one download:

In [ ]:
wh.download_zip(["melody.mid", "melody.xml", "melody.csv", "melody.json"], "my_markov_melody.zip")

## Where next?

- Combine your chains with GreasyPidgin's **envelopes** for dynamics, **L-systems** for
  structure, or **cellular automata** for rhythm.
- Learn chains from *several* MIDI files at once, by joining their pitch lists before calling
  `generateRulesFromList`.
- Use a chain's output as the input for the next one.